# 04 SpaceX Falcon 9 - Exploratory Data Analysis (SQL)

This notebook is **Step 04** in a multi-step end-to-end data science project that analyzes and predicts **first-stage landing success** for SpaceX Falcon 9 launches.

In this step, we load the labeled dataset produced in **Step 03** into a lightweight SQLite database and use **SQL** to explore patterns across launch sites, orbits, payload mass, and time.

**Pipeline overview:**

- **Step 01:** Collect launch data from the SpaceX REST API and create an initial modeling dataset.

- **Step 02:** Scrape a *fixed Wikipedia revision* of Falcon 9 & Falcon Heavy launch tables and export a clean CSV for supplementary analysis.

- **Step 03:** Clean and engineer features, create the landing success label (`Class`) and export the modeling dataset.

- **Step 04 (this notebook):** Load the labeled dataset into SQLite and answer exploratory questions using SQL.

- **Next steps:** Visual EDA, interactive geospatial mapping, dashboarding, and machine learning modeling.

**Input:** `../data/processed/03_dataset_part_2.csv` (Step 03)

**Outputs:**

- `../data/processed/04_spacex_launches.db` (SQLite database for ad-hoc querying)

- `../data/processed/04_launch_site_success_rate.csv` (launch-site level success summary)

## Notebook sections

**1. Setup**

**2. Load labeled dataset from Step 03**

**3. Build the analysis table**

**4. Exploratory SQL questions**

**5. Export artifacts**

---

## 1. Setup
We load the processed CSV from step 03, create a local SQLite database in `../data/processed/`, and expose a small helper for running SQL queries.

In [1]:
from pathlib import Path

import sqlite3
import pandas as pd

# Paths
PROCESSED_DIR = Path('../data/processed')
INPUT_CSV = PROCESSED_DIR / '03_dataset_part_2.csv'
DB_PATH = PROCESSED_DIR / '04_spacex_launches.db'

# Ensure output directory exists
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

# SQLite connection
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row

def q(sql: str, params = None) -> pd.DataFrame:
    """Run an SQL query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, con, params = params)

## 2. Load labeled dataset

We load the cleaned + labeled dataset from Step 03 that includes the binary target `Class` (1 = successful landing, 0 = unsuccessful/unknown).

In [2]:
df = pd.read_csv(INPUT_CSV)

# Write to a staging table
df.to_sql('spacex_tbl', con, if_exists = 'replace', index = False, method = 'multi')

df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


## 3. Build the analysis table

The course notebook creates a second table that excludes blank `Date` rows.
We keep the same idea here: `spacex_table` is the clean table used by all queries below.

In [3]:
# Drop analysis table if it already exists
con.execute('DROP TABLE IF EXISTS spacex_table;')
con.commit()

In [4]:
# Create the analysis table (exclude records without a date)
con.execute("""
            CREATE TABLE spacex_table AS
            SELECT *
            FROM spacex_tbl
            WHERE "Date" IS NOT NULL AND TRIM("Date") <> "";
""")

con.commit()

# Helpful indexes for repeated filtering
con.execute('CREATE INDEX IF NOT EXISTS idx_spacex_date ON spacex_table("Date");')
con.execute('CREATE INDEX IF NOT EXISTS idx_spacex_launch_site ON spacex_table("LaunchSite");')
con.execute('CREATE INDEX IF NOT EXISTS idx_spacex_orbit ON spacex_table("Orbit");')
con.execute('CREATE INDEX IF NOT EXISTS idx_spacex_class ON spacex_table("Class");')
con.commit()

q('SELECT COUNT(*) AS n_rows FROM spacex_table;')

,n_rows
0,90


## 4. Exploratory SQL Questions
Below are targeted SQL queries to understand how landing success varies by **launch site**, **orbit**, **payload mass**, and **time**.

### 4.1 Unique launch sites in the dataset

In [5]:
q("""
    SELECT DISTINCT "LaunchSite" AS launch_site
    FROM spacex_table
    ORDER BY launch_site;
""")

,launch_site
0,CCSFS SLC 40
1,KSC LC 39A
2,VAFB SLC 4E


### 4.2 Launche sites starting with `CC`

In [6]:
q("""
  SELECT DISTINCT "LaunchSite" AS launch_site
  FROM spacex_table
  WHERE "LaunchSite" LIKE 'CC%'
  ORDER BY launch_site;
""")


,launch_site
0,CCSFS SLC 40


### 4.3 Total payload mass by launch site
This helps compare how launch sites differ in the amount of payload they tend to support.

In [7]:
q("""
  SELECT
    "LaunchSite" AS launch_site,
    ROUND(SUM("PayloadMass"), 1) AS total_payload_kg,
    COUNT(*) AS n_launches
  FROM spacex_table
  GROUP BY "LaunchSite"
  ORDER BY total_payload_kg DESC;
""")

,launch_site,total_payload_kg,n_launches
0,CCSFS SLC 40,305987.2,55
1,KSC LC 39A,168179.1,22
2,VAFB SLC 4E,76953.0,13


### 4.4 Average payload mass by orbit
Orbit is one of the strongest candidates for explaining mission configuration differences.

In [8]:
q("""
  SELECT
    "Orbit" AS orbit,
    ROUND(AVG("PayloadMass"), 1) AS avg_payload_kg,
    COUNT(*) AS n_launches
  FROM spacex_table
  GROUP BY "Orbit"
  ORDER BY avg_payload_kg DESC;
""")

,orbit,avg_payload_kg,n_launches
0,VLEO,15428.6,14
1,PO,7583.7,9
2,GEO,6123.5,1
3,SO,6123.5,1
4,GTO,5012.0,27
5,MEO,3987.0,3
6,LEO,3890.8,7
7,ISS,3279.9,21
8,SSO,2060.0,5
9,ES-L1,570.0,1


#### 4.5 First observed successful landing date

We treat `Class = 1` as a successful first-stage landing.

In [9]:
q("""
  SELECT MIN("Date") AS first_success_date
  FROM spacex_table
  WHERE "Class" = 1;
""")

,first_success_date
0,2014-04-18


### 4.6 Launch count by Falcon 9 Block (booster revision)

In [10]:
q("""
  SELECT "Block", COUNT(*) AS n_launches
  FROM spacex_table
  GROUP BY "Block"
  ORDER BY n_launches DESC;
""")

,Block,n_launches
0,5.0,39
1,1.0,19
2,3.0,15
3,4.0,11
4,2.0,6


### 4.7 Success vs. failure counts
A quick view of class balance in the dataset.

In [11]:
q("""
  SELECT
    CASE WHEN class = 1 THEN 'success' ELSE 'failure' END AS landing_result,
    COUNT(*) AS n
  FROM spacex_table
  GROUP BY landing_result
  ORDER BY n DESC;
""")

,landing_result,n
0,success,60
1,failure,30


### 4.8 Block version(s) that carried the maximum payload

In [12]:
q("""
  SELECT DISTINCT "Block", MAX("PayloadMass") AS max_payload
  FROM spacex_table
  GROUP BY "Block"
  ORDER BY max_payload DESC;
""")

,Block,max_payload
0,5.0,15600.000000
1,4.0,9600.000000
2,3.0,9600.000000
3,1.0,6123.547647
4,2.0,5300.000000


### 4.9 Failed drone-ship landings in 2015 (month level)
SQLite doesn't have a built-in 'month name' function, so we extract the month using `substr(date, 6, 2)`.

In [13]:
q("""
  SELECT
    substr("Date", 6, 2) AS month,
    "Date" AS date,
    "LaunchSite" AS launch_site,
    "Orbit" AS orbit,
    "Outcome" AS landing_outcome,
    "PayloadMass" AS payload_mass_kg
  FROM spacex_table
  WHERE substr("Date", 1, 4) = '2015'
    AND "Outcome" LIKE '%ASDS%'
    AND "Class" = 0
  ORDER BY "Date";
""")

,month,date,launch_site,orbit,landing_outcome,payload_mass_kg
0,01,2015-01-10,CCSFS SLC 40,ISS,False ASDS,2395.0
1,04,2015-04-14,CCSFS SLC 40,ISS,False ASDS,1898.0
2,06,2015-06-28,CCSFS SLC 40,ISS,None ASDS,2477.0


### 4.10 Landing outcome frequency within a historical window

Counts each `landing_outcome` between 2010-06-04 and 2017-03-20 and ranks them in descending order.

In [14]:
q("""
  SELECT "Outcome" as landing_outcome, COUNT(*) AS count
  FROM spacex_table
  WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
  GROUP BY landing_outcome
  ORDER BY count DESC;
""")

,landing_outcome,count
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2


## Quick takeaways (SQL EDA)

A few patterns worth carrying into the visualization + modeling steps:

- Launces are concentrated across a small set of launch sites.

- Orbit and payload mass vary meaningfully across missions, and are likely informative features.

- Landing outcomes include multiple recovery modes (e.g. ASDS vs RTLS), which can be analyzed separately.

## 5. Export artifacts

Two artifacts are written to `../data/processed`:

- `04_spacex_launches.db` SQLite database containing `spacex_tbl` (raw import) and `spacex_table` (analysis table)

- `04_launch_site_success_rate.csv` Launch-site level success summary (useful for later reporting/plots)

In [15]:
launch_site_summary = q("""
SELECT
    "LaunchSite" AS launch_site,
    COUNT(*) AS n_launches,
    SUM("Class") AS n_success,
    ROUND(1.0 * SUM("Class") / COUNT(*), 3) AS success_rate
FROM spacex_table
GROUP BY launch_site
ORDER BY success_rate DESC, n_launches DESC;
""")

out_csv = PROCESSED_DIR / '04_launch_site_success_rate.csv'
launch_site_summary.to_csv(out_csv, index = False)

launch_site_summary

,launch_site,n_launches,n_success,success_rate
0,KSC LC 39A,22,17,0.773
1,VAFB SLC 4E,13,10,0.769
2,CCSFS SLC 40,55,33,0.600


In [16]:
# Close the SQLite connection
con.close()

## 6. Assumptions

- Uses Step 03 output `../data/processed/03_dataset_part_2.csv` as the source dataset for SQL EDA.

- Recreates the SQLite table (spacex_table) from the CSV each run (no dependency on a pre-existing DB).

- `Class` is binary: 1 = successful landing, 0 = unsuccessful.

- Rows with missing/blank Date are excluded when building the table; each row is treated as one launch record for counts/rates.

- Results are descriptive (not causal); small group sizes can make some success rates noisy.

---